In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
df = spark.read.format('parquet').load('abfss://bronze@strdatabricks.dfs.core.windows.net/orders')

In [0]:
df.show(n=5)

+--------+-----------+----------+----------+--------+------------+-------------+
|order_id|customer_id|product_id|order_date|quantity|total_amount|_rescued_data|
+--------+-----------+----------+----------+--------+------------+-------------+
|  O00001|     C00710|     P0159|2023-03-22|       3|     2022.87|         NULL|
|  O00002|     C00954|     P0036|2023-06-30|       2|     3560.74|         NULL|
|  O00003|     C01578|     P0427|2023-11-06|       3|     5903.52|         NULL|
|  O00004|     C00962|     P0332|2024-02-27|       3|     4107.99|         NULL|
|  O00005|     C00156|     P0038|2024-10-13|       5|     5784.95|         NULL|
+--------+-----------+----------+----------+--------+------------+-------------+
only showing top 5 rows


In [0]:
df = df.withColumnRenamed("_rescued_data","rescued_data")
df.show(5)

+--------+-----------+----------+----------+--------+------------+------------+
|order_id|customer_id|product_id|order_date|quantity|total_amount|rescued_data|
+--------+-----------+----------+----------+--------+------------+------------+
|  O00001|     C00710|     P0159|2023-03-22|       3|     2022.87|        NULL|
|  O00002|     C00954|     P0036|2023-06-30|       2|     3560.74|        NULL|
|  O00003|     C01578|     P0427|2023-11-06|       3|     5903.52|        NULL|
|  O00004|     C00962|     P0332|2024-02-27|       3|     4107.99|        NULL|
|  O00005|     C00156|     P0038|2024-10-13|       5|     5784.95|        NULL|
+--------+-----------+----------+----------+--------+------------+------------+
only showing top 5 rows


In [0]:
df = df.drop("rescued_data") 

In [0]:
df = df.withColumn("order_date", to_timestamp(col("order_date")))
df.show(5)

+--------+-----------+----------+-------------------+--------+------------+
|order_id|customer_id|product_id|         order_date|quantity|total_amount|
+--------+-----------+----------+-------------------+--------+------------+
|  O00001|     C00710|     P0159|2023-03-22 00:00:00|       3|     2022.87|
|  O00002|     C00954|     P0036|2023-06-30 00:00:00|       2|     3560.74|
|  O00003|     C01578|     P0427|2023-11-06 00:00:00|       3|     5903.52|
|  O00004|     C00962|     P0332|2024-02-27 00:00:00|       3|     4107.99|
|  O00005|     C00156|     P0038|2024-10-13 00:00:00|       5|     5784.95|
+--------+-----------+----------+-------------------+--------+------------+
only showing top 5 rows


In [0]:
df = df.withColumn("year", year(col("order_date")))
df.show(5)

+--------+-----------+----------+-------------------+--------+------------+----+
|order_id|customer_id|product_id|         order_date|quantity|total_amount|year|
+--------+-----------+----------+-------------------+--------+------------+----+
|  O00001|     C00710|     P0159|2023-03-22 00:00:00|       3|     2022.87|2023|
|  O00002|     C00954|     P0036|2023-06-30 00:00:00|       2|     3560.74|2023|
|  O00003|     C01578|     P0427|2023-11-06 00:00:00|       3|     5903.52|2023|
|  O00004|     C00962|     P0332|2024-02-27 00:00:00|       3|     4107.99|2024|
|  O00005|     C00156|     P0038|2024-10-13 00:00:00|       5|     5784.95|2024|
+--------+-----------+----------+-------------------+--------+------------+----+
only showing top 5 rows


In [0]:
df = df.withColumn("flag", dense_rank().over(
    Window.partitionBy("customer_id")
    .orderBy(desc("total_amount"),desc("order_date"))))
df.show(5)

+--------+-----------+----------+-------------------+--------+------------+----+----+
|order_id|customer_id|product_id|         order_date|quantity|total_amount|year|flag|
+--------+-----------+----------+-------------------+--------+------------+----+----+
|  O05531|     C00001|     P0456|2023-09-10 00:00:00|       5|     9524.35|2023|   1|
|  O08359|     C00001|     P0338|2023-03-09 00:00:00|       1|      1844.1|2023|   2|
|  O00186|     C00001|     P0126|2023-02-18 00:00:00|       1|     1431.12|2023|   3|
|  O02770|     C00001|     P0019|2023-12-09 00:00:00|       2|      461.12|2023|   4|
|  O05319|     C00002|     P0028|2023-05-12 00:00:00|       4|     6731.64|2023|   1|
+--------+-----------+----------+-------------------+--------+------------+----+----+
only showing top 5 rows


In [0]:
df = df.withColumn('rank_flag', rank().over(
    Window.partitionBy("year")
    .orderBy(desc("total_amount"))))
df.show()


+--------+-----------+----------+-------------------+--------+------------+----+----+---------+
|order_id|customer_id|product_id|         order_date|quantity|total_amount|year|flag|rank_flag|
+--------+-----------+----------+-------------------+--------+------------+----+----+---------+
|  O03502|     C00805|     P0498|2023-09-24 00:00:00|       5|      9952.9|2023|   1|        1|
|  O03660|     C01001|     P0498|2023-10-08 00:00:00|       5|      9952.9|2023|   1|        1|
|  O00957|     C01449|     P0498|2023-10-05 00:00:00|       5|      9952.9|2023|   1|        1|
|  O01765|     C01515|     P0498|2023-05-11 00:00:00|       5|      9952.9|2023|   1|        1|
|  O06790|     C01819|     P0498|2023-02-27 00:00:00|       5|      9952.9|2023|   1|        1|
|  O07763|     C00471|     P0440|2023-04-12 00:00:00|       5|     9916.25|2023|   1|        6|
|  O03989|     C00631|     P0440|2023-11-20 00:00:00|       5|     9916.25|2023|   1|        6|
|  O08261|     C00988|     P0165|2023-11

In [0]:
df = df.withColumn("row_flag",row_number().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
df.show()

+--------+-----------+----------+-------------------+--------+------------+----+----+---------+--------+
|order_id|customer_id|product_id|         order_date|quantity|total_amount|year|flag|rank_flag|row_flag|
+--------+-----------+----------+-------------------+--------+------------+----+----+---------+--------+
|  O03502|     C00805|     P0498|2023-09-24 00:00:00|       5|      9952.9|2023|   1|        1|       1|
|  O03660|     C01001|     P0498|2023-10-08 00:00:00|       5|      9952.9|2023|   1|        1|       2|
|  O00957|     C01449|     P0498|2023-10-05 00:00:00|       5|      9952.9|2023|   1|        1|       3|
|  O01765|     C01515|     P0498|2023-05-11 00:00:00|       5|      9952.9|2023|   1|        1|       4|
|  O06790|     C01819|     P0498|2023-02-27 00:00:00|       5|      9952.9|2023|   1|        1|       5|
|  O07763|     C00471|     P0440|2023-04-12 00:00:00|       5|     9916.25|2023|   1|        6|       6|
|  O03989|     C00631|     P0440|2023-11-20 00:00:00|  

## Classes - OPPS


In [0]:
class windows:

    def dense_rank(self,df):
        df_new = df.withColumn("DenseRank_flag", dense_rank().over(
            Window.partitionBy("year")
            .orderBy(desc("total_amount"))))
        return df_new
    
    def rank(self,df):
        df_new = df.withColumn("Rank_flag", rank().over(
            Window.partitionBy("year")
            .orderBy(desc("total_amount"))))
        return df_new
    def row_number(self,df):
        df_new = df.withColumn("RowNumber_flag", row_number().over(
            Window.partitionBy("year")
            .orderBy(desc("total_amount"))))
        return df_new
df_new = windows().dense_rank(df)
df_new = windows().rank(df_new)
df_new = windows().row_number(df_new)
df_new.show()

+--------+-----------+----------+-------------------+--------+------------+----+----+---------+--------+--------------+--------------+
|order_id|customer_id|product_id|         order_date|quantity|total_amount|year|flag|Rank_flag|row_flag|DenseRank_flag|RowNumber_flag|
+--------+-----------+----------+-------------------+--------+------------+----+----+---------+--------+--------------+--------------+
|  O03502|     C00805|     P0498|2023-09-24 00:00:00|       5|      9952.9|2023|   1|        1|       1|             1|             1|
|  O03660|     C01001|     P0498|2023-10-08 00:00:00|       5|      9952.9|2023|   1|        1|       2|             1|             2|
|  O00957|     C01449|     P0498|2023-10-05 00:00:00|       5|      9952.9|2023|   1|        1|       3|             1|             3|
|  O01765|     C01515|     P0498|2023-05-11 00:00:00|       5|      9952.9|2023|   1|        1|       4|             1|             4|
|  O06790|     C01819|     P0498|2023-02-27 00:00:00|  

## Data Writing


In [0]:
df.write.format("delta").mode("overwrite").save("abfss://silver@strdatabricks.dfs.core.windows.net/orders")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS datapipeline.silver.orders
USING DELTA
LOCATION 'abfss://silver@strdatabricks.dfs.core.windows.net/orders'